# Sprint 0: Kiểm tra môi trường Kaggle

Notebook này kiểm tra rằng Kaggle, repo và W&B đã sẵn sàng trước khi viết code thật. Nó **không** chạy RanDumb.

**Chuẩn bị trên Kaggle trước khi chạy:**
1. **Settings → Accelerator:** GPU T4 x2.
2. **Settings → Internet:** On (cần xác minh số điện thoại).
3. **Add-ons → Secrets:** thêm `WANDB_API_KEY` và bật cho notebook này. Không dán key vào code.
4. Sửa `WANDB_ENTITY` ở ô đầu tiên thành tên team W&B của nhóm.

**Kết quả cần lấy về:**
- Danh sách `pkg==version` ở ô 3 để ghim `requirements.txt`.
- Một run có thẻ `smoke` trong W&B project của nhóm.
- Bảng kiểm tra dataset ở ô 7, dùng cho `data.py` ở Sprint 1.
- Số đo `float32` và `float64` ở ô 6, chỉ để tham khảo cho Sprint 2.

In [ ]:
# Ô 1. Cấu hình
WANDB_ENTITY = "TEN-TEAM-WANDB"  # đổi thành tên team W&B của nhóm
WANDB_PROJECT = "randumb-repro"
REPO_URL = "https://github.com/24122039Kien/RanDumb-reproduction.git"
REPO_DIR = "/kaggle/working/RanDumb-reproduction"
SEED = 0

In [ ]:
# Ô 2. Phần cứng
import os
import platform
import shutil
import subprocess
import sys

import psutil

env = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "cpu_count": os.cpu_count(),
    "ram_gb": round(psutil.virtual_memory().total / 2**30, 1),
    "disk_free_working_gb": round(shutil.disk_usage("/kaggle/working").free / 2**30, 1),
}
for key, value in env.items():
    print(f"{key:>22}: {value}")
print()
subprocess.run(["nvidia-smi"], check=False)

In [ ]:
# Ô 3. Phiên bản thư viện, cài thêm nếu thiếu
from importlib import metadata

PACKAGES = ["numpy", "scipy", "scikit-learn", "torch", "wandb", "datasets", "timm", "psutil", "pytest"]


def installed_versions(names):
    """Trả về {tên gói: phiên bản}, None nếu chưa cài."""
    versions = {}
    for name in names:
        try:
            versions[name] = metadata.version(name)
        except metadata.PackageNotFoundError:
            versions[name] = None
    return versions


missing = [name for name, version in installed_versions(PACKAGES).items() if version is None]
if missing:
    print("Đang cài:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

versions = installed_versions(PACKAGES)
env.update({f"version_{name}": version for name, version in versions.items()})
print("Dán các dòng dưới đây vào requirements.txt:")
for name, version in versions.items():
    print(f"{name}=={version}")

In [ ]:
# Ô 4. Clone repo của nhóm, kiểm tra import và test
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

env["repo_commit"] = subprocess.run(
    ["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
print("Commit:", env["repo_commit"])

src_dir = os.path.join(REPO_DIR, "src")
if os.path.isdir(os.path.join(src_dir, "randumb")):
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)
    import randumb  # noqa: F401
    print("Import randumb: OK")
    subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"], cwd=REPO_DIR, check=False)
else:
    print("Repo chưa có src/randumb: hãy push khung repo rồi chạy lại ô này.")

In [ ]:
# Ô 5. GPU và seed
import random

import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

env["seed"] = SEED
env["torch_cuda"] = torch.version.cuda
env["gpu_count"] = torch.cuda.device_count()
env["gpu_name"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
env["gpu_mem_gb"] = (
    round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1) if torch.cuda.is_available() else None
)
print({k: env[k] for k in ["torch_cuda", "gpu_count", "gpu_name", "gpu_mem_gb"]})
assert torch.cuda.is_available(), "Chưa bật GPU: Settings → Accelerator → GPU T4 x2"

In [ ]:
# Ô 6. Đo nhanh float32 và float64 trên GPU (chỉ để tham khảo, không thay Bước 3)
import time


def time_cuda(fn, reps=3):
    """Chạy fn một lần để làm nóng, rồi trả về số giây trung bình của reps lần."""
    fn()
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(reps):
        fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - start) / reps


bench = {}
n = 4096
for dtype in (torch.float32, torch.float64):
    name = str(dtype).replace("torch.", "")
    a = torch.randn(n, n, device="cuda", dtype=dtype)
    spd = a @ a.T / n + torch.eye(n, device="cuda", dtype=dtype)
    matmul_s = time_cuda(lambda: a @ a)
    bench[f"matmul_tflops_{name}"] = round(2 * n**3 / matmul_s / 1e12, 3)
    bench[f"cholesky_s_{name}"] = round(time_cuda(lambda: torch.linalg.cholesky(spd)), 4)
    del a, spd
    torch.cuda.empty_cache()

for key, value in bench.items():
    print(f"{key:>24}: {value}")

In [ ]:
# Ô 7. Kiểm tra nguồn dữ liệu trên Hugging Face, chỉ đọc metadata, không tải ảnh
from datasets import load_dataset_builder

try:
    from randumb.data import DATASETS

    specs = {key: (spec.hf_id, spec.num_train, spec.num_test) for key, spec in DATASETS.items()}
except ImportError:
    # Bản sao của randumb.data.DATASETS, dùng khi repo chưa có src/
    specs = {
        "mnist": ("ylecun/mnist", 60_000, 10_000),
        "cifar10": ("uoft-cs/cifar10", 50_000, 10_000),
        "cifar100": ("uoft-cs/cifar100", 50_000, 10_000),
        "tiny-imagenet": ("Maysee/tiny-imagenet", 100_000, 10_000),
        "mini-imagenet": ("timm/mini-imagenet", 50_000, 10_000),
    }

dataset_rows = []
for key, (hf_id, num_train, num_test) in specs.items():
    try:
        info = load_dataset_builder(hf_id).info
        splits = {name: split.num_examples for name, split in (info.splits or {}).items()}
        columns = list(info.features) if info.features else []
        status = "ok" if splits.get("train") == num_train else "kiểm tra lại số mẫu"
    except Exception as exc:  # ghi lại lỗi thay vì dừng notebook
        splits, columns, status = {}, [], f"lỗi: {type(exc).__name__}: {exc}"[:200]
    dataset_rows.append([key, hf_id, status, str(splits), str(columns)])
    print(f"{key:>14} | {status} | splits={splits} | columns={columns}")

In [ ]:
# Ô 8. Đăng nhập W&B bằng Kaggle Secrets
import wandb
from kaggle_secrets import UserSecretsClient

wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))

In [ ]:
# Ô 9. Log run smoke
run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    name=f"smoke-kaggle-seed{SEED}",
    group="smoke",
    job_type="smoke",
    tags=["smoke"],
    config=env,
)
run.log(bench)
run.log({"dataset_check": wandb.Table(columns=["key", "hf_id", "status", "splits", "columns"], data=dataset_rows)})
print("Run:", run.url)
run.finish()

## Sau khi chạy xong

- [ ] Mở link run ở **cửa sổ ẩn danh** để chắc chắn project xem được khi chưa đăng nhập.
- [ ] Dán danh sách phiên bản ở ô 3 vào `requirements.txt` và commit.
- [ ] Gửi bảng kiểm tra dataset ở ô 7 cho người làm `data.py`.
- [ ] Ghi số đo ở ô 6 vào ghi chú Sprint 2. Đây chưa phải số đo của pipeline RanDumb.